# D3 — Stronger Neo4j / Cypher Queries (Member 2)

Standalone notebook containing only the D3 deliverable: multi-hop graph expansion, subgraph selection/scoring, and the drop-in replacement for Member 1's `expand_by_shared_topics()`.

This assumes the graph already exists in Neo4j (built in `03_graph_build.ipynb`, D2). This notebook only **connects and queries** — it does not rebuild the graph.

## Setup — connect to the existing Neo4j graph

In [1]:
import pandas as pd
from neo4j import GraphDatabase

NEO4J_URI = "neo4j+s://31a35718.databases.neo4j.io"
NEO4J_USER = "31a35718"
NEO4J_PASSWORD = "4a5QbeBuQy47jPN0-tVX4TlGZPHsZQUKPASOtNSCp2g"
NEO4J_DATABASE = "31a35718"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]

try:
    test_result = run_cypher("RETURN 'Neo4j connection OK' AS status")
    print(test_result[0]["status"])
except Exception as e:
    print("Neo4j connection failed.")
    print("Check that the Aura instance is running (not paused) and that NEO4J_PASSWORD is correct.")
    raise e

Neo4j connection OK


In [2]:
# Pull paper_ids that already exist in the graph (built in D2) to use as
# seed papers for testing the D3 expansion queries below.

papers_query = "MATCH (p:Paper) RETURN p.paper_id AS paper_id, p.title AS title LIMIT 10"
papers = run_cypher(papers_query)

print(f"Loaded {len(papers)} sample papers from the existing graph.")
display(pd.DataFrame(papers))

Loaded 10 sample papers from the existing graph.


,paper_id,title
0,paper1,FlexSQL: Flexible Exploration and Execution Ma...
1,paper2,Reinforcement Learning for LLM-based Multi-Age...
2,paper3,mdok-style at SemEval-2026 Task 10: Finetuning...
3,paper4,mdok-style at SemEval-2026 Task 9: Finetuning ...
4,paper5,Fuzzy Fingerprinting Encoder Pre-trained Langu...
5,paper6,ContextualJailbreak: Evolutionary Red-Teaming ...
6,paper7,Mapping Discourse Reframing: A Multi-Layer Net...
7,paper8,"Synthetic Users, Real Differences: an Evaluati..."
8,paper9,Beating the Style Detector: Three Hours of Age...
9,paper10,Dependency Parsing Across the Resource Spectru...


**What's new vs. the D2 single-hop query:**
1. **Multi-hop topic expansion** — climbs the `SUBTOPIC_OF` hierarchy so siblings under the same parent topic (e.g. "Dense Retrieval" and "Hybrid Search" both under "Information Retrieval") are captured, not just exact-topic matches.
2. **Author paths, 2-hop** — co-author-of-co-author reasoning, not just direct shared authorship.
3. **Combined paper-to-paper path scoring** — one query walks topic + hierarchy + author + co-author paths together and aggregates per candidate, so papers connected by *multiple* independent paths score higher.
4. **Real subgraph selection** — every candidate is scored, ranked, and capped (`LIMIT`), instead of returning everything the graph happens to touch.
5. **Drop-in replacement function** for Member 1's `expand_by_shared_topics()` in `graphrag_executor.ipynb` — same input/output shape (`{paper_id, graph_score, reasons}`), but backed by live multi-hop Cypher instead of a flat CSV lookup.

In [3]:
# Driver and run_cypher() already set up above.
print("Connected and ready.")

Connected and ready.


## 1. Multi-hop topic expansion (climbs the SUBTOPIC_OF hierarchy)

In [4]:
TOPIC_EXPANSION_MULTIHOP = """
MATCH (seed:Paper {paper_id: $paper_id})

// Direct topic match (0-hop): same topic node
OPTIONAL MATCH (seed)-[sr:HAS_TOPIC]->(t:Topic)<-[tr:HAS_TOPIC]-(direct:Paper)
WHERE direct <> seed

// Sibling topic match (2-hop): seed's topic and candidate's topic
// share a parent in the SUBTOPIC_OF hierarchy, but are not the same topic
OPTIONAL MATCH (seed)-[:HAS_TOPIC]->(seed_topic:Topic)-[:SUBTOPIC_OF]->(parent:Topic)<-[:SUBTOPIC_OF]-(sibling_topic:Topic)<-[sib_r:HAS_TOPIC]-(sibling:Paper)
WHERE sibling <> seed
  AND sibling_topic <> seed_topic

WITH seed,
     collect(DISTINCT {
         paper_id:    direct.paper_id,
         topic:       t.name,
         confidence:  tr.confidence,
         hop_type:    "direct_topic",
         hop_distance: 0
     }) AS direct_matches,
     collect(DISTINCT {
         paper_id:    sibling.paper_id,
         topic:       sibling_topic.name,
         parent_topic: parent.name,
         confidence:  sib_r.confidence,
         hop_type:    "sibling_topic",
         hop_distance: 2
     }) AS sibling_matches

RETURN seed.paper_id AS seed_paper_id,
       direct_matches,
       sibling_matches
"""

seed_paper_id = papers[0]["paper_id"]
multihop_result = run_cypher(TOPIC_EXPANSION_MULTIHOP, {"paper_id": seed_paper_id})

print(f"Seed paper: {seed_paper_id}")
print(f"Direct topic matches: {len([m for m in multihop_result[0]['direct_matches'] if m['paper_id']])}")
print(f"Sibling topic matches (new in D3): {len([m for m in multihop_result[0]['sibling_matches'] if m['paper_id']])}")

display(pd.DataFrame([m for m in multihop_result[0]["sibling_matches"] if m["paper_id"]]))

Seed paper: paper1
Direct topic matches: 50
Sibling topic matches (new in D3): 210


,parent_topic,confidence,hop_distance,topic,paper_id,hop_type
0,Large Language Models,0.95,2,Reasoning,paper15,sibling_topic
1,Large Language Models,0.82,2,Reasoning,paper21,sibling_topic
2,Large Language Models,0.82,2,Reasoning,paper23,sibling_topic
3,Large Language Models,0.82,2,Reasoning,paper24,sibling_topic
4,Large Language Models,0.82,2,Reasoning,paper27,sibling_topic
...,...,...,...,...,...,...
205,Large Language Models,0.82,2,Code Generation,paper24,sibling_topic
206,Large Language Models,0.82,2,Code Generation,paper34,sibling_topic
207,Large Language Models,0.82,2,Code Generation,paper78,sibling_topic
208,Large Language Models,0.82,2,Code Generation,paper126,sibling_topic


## 2. Author collaboration paths, 2-hop (co-author-of-co-author)

In [5]:
AUTHOR_EXPANSION_MULTIHOP = """
MATCH (seed:Paper {paper_id: $paper_id})

// Direct shared author (1-hop)
OPTIONAL MATCH (seed)<-[:WROTE]-(a:Author)-[:WROTE]->(direct:Paper)
WHERE direct <> seed

// Co-author network (2-hop): authors who co-wrote a paper WITH one of the
// seed's authors, then look at what else those collaborators wrote
OPTIONAL MATCH (seed)<-[:WROTE]-(seed_author:Author)-[:WROTE]->(:Paper)<-[:WROTE]-(collaborator:Author)-[:WROTE]->(network:Paper)
WHERE network <> seed
  AND collaborator <> seed_author
  AND NOT (seed)<-[:WROTE]-(collaborator)

WITH seed,
     collect(DISTINCT {
         paper_id: direct.paper_id,
         author:   a.name,
         hop_type: "direct_author",
         hop_distance: 1
     }) AS direct_author_matches,
     collect(DISTINCT {
         paper_id:    network.paper_id,
         via_author:  seed_author.name,
         collaborator: collaborator.name,
         hop_type:    "coauthor_network",
         hop_distance: 2
     }) AS network_matches

RETURN seed.paper_id AS seed_paper_id,
       direct_author_matches,
       network_matches
"""

author_result = run_cypher(AUTHOR_EXPANSION_MULTIHOP, {"paper_id": seed_paper_id})

direct_n = len([m for m in author_result[0]["direct_author_matches"] if m["paper_id"]])
network_n = len([m for m in author_result[0]["network_matches"] if m["paper_id"]])
print(f"Direct shared-author matches: {direct_n}")
print(f"Co-author network matches (new in D3, 2-hop): {network_n}")

display(pd.DataFrame([m for m in author_result[0]["network_matches"] if m["paper_id"]]))

Direct shared-author matches: 0
Co-author network matches (new in D3, 2-hop): 0


""


## 3. Combined path scoring + subgraph selection

This is the core upgrade: instead of two separate unscored lists (topic matches, author matches), one query walks **all four path types** — direct topic, sibling topic, direct author, co-author network — and aggregates them **per candidate paper**. A paper connected by multiple independent paths (e.g. shares a topic *and* an author with the seed) scores higher than one connected by a single weak path.

The final `LIMIT $limit` is the actual **subgraph selection** step — we deliberately keep only the top-N ranked candidates rather than returning everything the graph touches.

In [6]:
SUBGRAPH_SELECTION_QUERY = """
MATCH (seed:Paper {paper_id: $paper_id})

// Path 1: direct topic
OPTIONAL MATCH (seed)-[:HAS_TOPIC]->(t1:Topic)<-[r1:HAS_TOPIC]-(c1:Paper)
WHERE c1 <> seed
WITH seed, collect(DISTINCT {paper_id: c1.paper_id, weight: coalesce(r1.confidence, 0.5), reason: "shared_topic: " + t1.name}) AS p1

// Path 2: sibling topic (via hierarchy)
OPTIONAL MATCH (seed)-[:HAS_TOPIC]->(st:Topic)-[:SUBTOPIC_OF]->(parent:Topic)<-[:SUBTOPIC_OF]-(sib:Topic)<-[r2:HAS_TOPIC]-(c2:Paper)
WHERE c2 <> seed AND sib <> st
WITH seed, p1, collect(DISTINCT {paper_id: c2.paper_id, weight: coalesce(r2.confidence, 0.5) * 0.5, reason: "sibling_topic: " + sib.name + " (parent: " + parent.name + ")"}) AS p2

// Path 3: direct shared author
OPTIONAL MATCH (seed)<-[:WROTE]-(a:Author)-[:WROTE]->(c3:Paper)
WHERE c3 <> seed
WITH seed, p1, p2, collect(DISTINCT {paper_id: c3.paper_id, weight: 1.0, reason: "shared_author: " + a.name}) AS p3

// Path 4: co-author network (2-hop)
OPTIONAL MATCH (seed)<-[:WROTE]-(sa:Author)-[:WROTE]->(:Paper)<-[:WROTE]-(collab:Author)-[:WROTE]->(c4:Paper)
WHERE c4 <> seed AND collab <> sa AND NOT (seed)<-[:WROTE]-(collab)
WITH seed, p1, p2, p3, collect(DISTINCT {paper_id: c4.paper_id, weight: 0.4, reason: "coauthor_network: " + sa.name + " -> " + collab.name}) AS p4

// Flatten and aggregate per candidate paper
WITH seed, p1 + p2 + p3 + p4 AS all_paths
UNWIND all_paths AS path
WITH seed, path
WHERE path.paper_id IS NOT NULL

WITH seed,
     path.paper_id AS candidate_id,
     sum(path.weight) AS path_score,
     count(path) AS path_count,
     collect(DISTINCT path.reason) AS reasons

MATCH (candidate:Paper {paper_id: candidate_id})

RETURN candidate.paper_id AS paper_id,
       candidate.title AS title,
       round(path_score, 4) AS graph_score,
       path_count,
       reasons
ORDER BY graph_score DESC, path_count DESC
LIMIT $limit
"""

def select_subgraph(paper_id, limit=5, min_score=0.0):
    """Score every connected candidate by combined path strength and
    keep only the top `limit` above `min_score`. This is the subgraph
    selection step the rubric asks for."""
    results = run_cypher(SUBGRAPH_SELECTION_QUERY, {"paper_id": paper_id, "limit": limit})
    return [r for r in results if r["graph_score"] >= min_score]


subgraph_df = pd.DataFrame(select_subgraph(seed_paper_id, limit=8))
print(f"Selected subgraph for seed paper {seed_paper_id}:")
display(subgraph_df)

Selected subgraph for seed paper paper1:


,paper_id,title,graph_score,path_count,reasons
0,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,3.475,6,"[shared_topic: Code Generation, shared_topic: ..."
1,paper24,MolViBench: Evaluating LLMs on Molecular Vibe ...,2.870,5,"[shared_topic: Code Generation, shared_topic: ..."
2,paper127,SciResearcher: Scaling Deep Research Agents fo...,2.655,5,"[shared_topic: Reasoning, sibling_topic: Reaso..."
3,paper43,Maistros: A Greek Large Language Model Adapted...,2.245,4,"[shared_topic: Reasoning, sibling_topic: Reaso..."
4,paper165,Do Reasoning Vision-Language Models Inversely ...,2.245,4,"[shared_topic: Reasoning, sibling_topic: Reaso..."
5,paper68,Medmarks: A Comprehensive Open-Source LLM Benc...,2.115,4,"[shared_topic: Reasoning, sibling_topic: Reaso..."
6,paper47,RMGAP: Benchmarking the Generalization of Rewa...,2.115,4,"[shared_topic: Reasoning, sibling_topic: Reaso..."
7,paper126,Feedback-Normalized Developer Memory for Reinf...,2.115,4,"[shared_topic: Code Generation, sibling_topic:..."


## 4. Drop-in replacement for `expand_by_shared_topics()`

Member 1's `graphrag_executor.ipynb` currently expands the graph by reading `neo4j_topic_assignments.csv` with pandas and matching on exact topic name only (single-hop, topic-only, no author signal).

`expand_by_shared_topics_cypher()` below takes the **same inputs** (`seed_paper_ids`, `limit`) and returns the **same output shape** (`{paper_id, graph_score, reasons}`), so it can replace `expand_by_shared_topics()` in the executor with no other code changes — but now backed by live multi-hop Cypher (topic + hierarchy + author + co-author paths) instead of a flat CSV lookup.

In [7]:
def expand_by_shared_topics_cypher(seed_paper_ids, limit=5):
    """
    Multi-hop, multi-path Neo4j subgraph expansion.
    Drop-in replacement for graphrag_executor.ipynb's pandas/CSV-based
    expand_by_shared_topics(seed_paper_ids, limit).

    Returns: [{"paper_id": str, "graph_score": float, "reasons": [str]}]
    (identical shape to the function it replaces)
    """
    if not seed_paper_ids:
        return []

    merged = {}
    for pid in seed_paper_ids:
        candidates = select_subgraph(pid, limit=limit * 2)  # over-fetch, merge, trim
        for c in candidates:
            cid = c["paper_id"]
            if cid in seed_paper_ids:
                continue
            if cid not in merged:
                merged[cid] = {"paper_id": cid, "graph_score": 0.0, "reasons": []}
            merged[cid]["graph_score"] += c["graph_score"]
            for r in c["reasons"]:
                if r not in merged[cid]["reasons"]:
                    merged[cid]["reasons"].append(r)

    expanded = sorted(merged.values(), key=lambda x: x["graph_score"], reverse=True)
    return expanded[:limit]


# Test with a multi-paper seed set, same as graphrag_executor.ipynb would pass
test_seed_ids = [p["paper_id"] for p in papers[:3]]
drop_in_result = expand_by_shared_topics_cypher(test_seed_ids, limit=5)

print(f"Seed papers: {test_seed_ids}")
print(f"\nDrop-in replacement result ({len(drop_in_result)} expanded papers):")
display(pd.DataFrame(drop_in_result))

Seed papers: ['paper1', 'paper2', 'paper3']

Drop-in replacement result (5 expanded papers):


,paper_id,graph_score,reasons
0,paper140,6.820,"[shared_topic: Reasoning, sibling_topic: Reaso..."
1,paper126,5.180,"[shared_topic: Code Generation, sibling_topic:..."
2,paper68,5.180,"[shared_topic: Reasoning, sibling_topic: Reaso..."
3,paper169,5.180,"[shared_topic: Reasoning, sibling_topic: Reaso..."
4,paper43,3.605,"[shared_topic: Reasoning, sibling_topic: Reaso..."


## 5. D2 vs D3 comparison

Quick before/after on the same seed paper: D2's single-hop query vs D3's multi-hop, multi-path, scored subgraph selection.

In [8]:
# D2 (single-hop, unscored) — reusing the exact query from the cell above
d2_query = """
MATCH (seed:Paper {paper_id: $paper_id})
OPTIONAL MATCH (seed)-[sr:HAS_TOPIC]->(t:Topic)<-[tr:HAS_TOPIC]-(related_by_topic:Paper)
WHERE related_by_topic <> seed
OPTIONAL MATCH (seed)<-[:WROTE]-(a:Author)-[:WROTE]->(related_by_author:Paper)
WHERE related_by_author <> seed
RETURN
    count(DISTINCT related_by_topic.paper_id) AS d2_topic_matches,
    count(DISTINCT related_by_author.paper_id) AS d2_author_matches
"""
d2_counts = run_cypher(d2_query, {"paper_id": seed_paper_id})[0]

d3_subgraph = select_subgraph(seed_paper_id, limit=20)

comparison_df = pd.DataFrame([
    {"version": "D2 (single-hop, unscored)",
     "topic_candidates": d2_counts["d2_topic_matches"],
     "author_candidates": d2_counts["d2_author_matches"],
     "scored_subgraph": "no",
     "multi_path_aggregation": "no"},
    {"version": "D3 (multi-hop, scored)",
     "topic_candidates": "direct + sibling (hierarchy)",
     "author_candidates": "direct + 2-hop co-author network",
     "scored_subgraph": "yes (path_score, ranked, LIMIT)",
     "multi_path_aggregation": f"yes ({len(d3_subgraph)} candidates after selection)"},
])
display(comparison_df)

,version,topic_candidates,author_candidates,scored_subgraph,multi_path_aggregation
0,"D2 (single-hop, unscored)",48,0,no,no
1,"D3 (multi-hop, scored)",direct + sibling (hierarchy),direct + 2-hop co-author network,"yes (path_score, ranked, LIMIT)",yes (20 candidates after selection)


## 6. Structured path explanations (explicit `path_type` / `path` / `graph_score`)

`reasons` in Sections 3-4 is a flat list of human-readable strings (e.g. `"shared_topic: RAG"`) — good for display, but not something Member 1's executor could act on programmatically if it ever needed to (e.g. filtering by path type, or rendering an explicit hop-by-hop path).

This section adds a **structured** version alongside the existing one. Each path a candidate is connected by becomes its own object with:
- `path_type` — `"shared_topic"`, `"sibling_topic"`, `"shared_author"`, or `"coauthor_network"`
- `path` — the literal hop sequence, e.g. `["paper12", "RAG", "paper45"]`
- `weight` — that path's individual contribution to the aggregate `graph_score`

A candidate connected by multiple path types (e.g. shared topic **and** shared author) gets multiple entries in `paths`, not one merged string — `graph_score` stays the single aggregate Member 1 sorts/blends on; `paths` is for explanation/display.

This does **not** replace `select_subgraph()` or `expand_by_shared_topics_cypher()` from Sections 3-4 — those keep working as-is. This is an additive, richer view of the same underlying data.

In [9]:
SUBGRAPH_SELECTION_STRUCTURED_QUERY = """
MATCH (seed:Paper {paper_id: $paper_id})

// Path 1: direct topic
OPTIONAL MATCH (seed)-[:HAS_TOPIC]->(t1:Topic)<-[r1:HAS_TOPIC]-(c1:Paper)
WHERE c1 <> seed
WITH seed, collect(DISTINCT {
    paper_id: c1.paper_id,
    weight: coalesce(r1.confidence, 0.5),
    path_type: "shared_topic",
    path: [seed.paper_id, t1.name, c1.paper_id]
}) AS p1

// Path 2: sibling topic (via hierarchy)
OPTIONAL MATCH (seed)-[:HAS_TOPIC]->(st:Topic)-[:SUBTOPIC_OF]->(parent:Topic)<-[:SUBTOPIC_OF]-(sib:Topic)<-[r2:HAS_TOPIC]-(c2:Paper)
WHERE c2 <> seed AND sib <> st
WITH seed, p1, collect(DISTINCT {
    paper_id: c2.paper_id,
    weight: coalesce(r2.confidence, 0.5) * 0.5,
    path_type: "sibling_topic",
    path: [seed.paper_id, st.name, parent.name, sib.name, c2.paper_id]
}) AS p2

// Path 3: direct shared author
OPTIONAL MATCH (seed)<-[:WROTE]-(a:Author)-[:WROTE]->(c3:Paper)
WHERE c3 <> seed
WITH seed, p1, p2, collect(DISTINCT {
    paper_id: c3.paper_id,
    weight: 1.0,
    path_type: "shared_author",
    path: [seed.paper_id, a.name, c3.paper_id]
}) AS p3

// Path 4: co-author network (2-hop)
OPTIONAL MATCH (seed)<-[:WROTE]-(sa:Author)-[:WROTE]->(:Paper)<-[:WROTE]-(collab:Author)-[:WROTE]->(c4:Paper)
WHERE c4 <> seed AND collab <> sa AND NOT (seed)<-[:WROTE]-(collab)
WITH seed, p1, p2, p3, collect(DISTINCT {
    paper_id: c4.paper_id,
    weight: 0.4,
    path_type: "coauthor_network",
    path: [seed.paper_id, sa.name, collab.name, c4.paper_id]
}) AS p4

// Flatten and aggregate per candidate paper
WITH seed, p1 + p2 + p3 + p4 AS all_paths
UNWIND all_paths AS path_item
WITH seed, path_item
WHERE path_item.paper_id IS NOT NULL

WITH seed,
     path_item.paper_id AS candidate_id,
     sum(path_item.weight) AS graph_score,
     collect(DISTINCT {
         path_type: path_item.path_type,
         path:      path_item.path,
         weight:    path_item.weight
     }) AS paths

MATCH (candidate:Paper {paper_id: candidate_id})

RETURN candidate.paper_id AS paper_id,
       candidate.title AS title,
       round(graph_score, 4) AS graph_score,
       paths
ORDER BY graph_score DESC
LIMIT $limit
"""


def select_subgraph_structured(paper_id, limit=5, min_score=0.0):
    """
    Same scoring/ranking as select_subgraph(), but each candidate's
    `paths` field is a list of structured objects:
        {"path_type": str, "path": [hop, hop, ...], "weight": float}
    instead of a flat list of pre-formatted strings.

    Use this when Member 1 (or the report/poster) needs to render an
    explicit hop-by-hop explanation, e.g. "paper12 -> RAG -> paper45".
    """
    results = run_cypher(SUBGRAPH_SELECTION_STRUCTURED_QUERY, {"paper_id": paper_id, "limit": limit})
    return [r for r in results if r["graph_score"] >= min_score]


structured_subgraph = select_subgraph_structured(seed_paper_id, limit=5)

print(f"Structured subgraph for seed paper {seed_paper_id}:\n")
for row in structured_subgraph:
    print(f"{row['paper_id']}  (graph_score={row['graph_score']})")
    for p in row["paths"]:
        arrow_path = " -> ".join(p["path"])
        print(f"    [{p['path_type']}, weight={p['weight']}]  {arrow_path}")
    print()

Structured subgraph for seed paper paper1:

paper34  (graph_score=4.295)
    [shared_topic, weight=0.82]  paper1 -> Code Generation -> paper34
    [shared_topic, weight=0.95]  paper1 -> Reasoning -> paper34
    [sibling_topic, weight=0.475]  paper1 -> Code Generation -> Large Language Models -> Reasoning -> paper34
    [sibling_topic, weight=0.41]  paper1 -> Code Generation -> Large Language Models -> Fine-Tuning -> paper34
    [sibling_topic, weight=0.41]  paper1 -> Code Generation -> Large Language Models -> LLM Evaluation -> paper34
    [sibling_topic, weight=0.41]  paper1 -> Reasoning -> Large Language Models -> Code Generation -> paper34
    [sibling_topic, weight=0.41]  paper1 -> Reasoning -> Large Language Models -> Fine-Tuning -> paper34
    [sibling_topic, weight=0.41]  paper1 -> Reasoning -> Large Language Models -> LLM Evaluation -> paper34

paper127  (graph_score=3.885)
    [shared_topic, weight=0.95]  paper1 -> Reasoning -> paper127
    [sibling_topic, weight=0.475]  paper

In [10]:
# Flat table view for quick inspection / export
flat_rows = []
for row in structured_subgraph:
    for p in row["paths"]:
        flat_rows.append({
            "paper_id":   row["paper_id"],
            "title":      row["title"],
            "graph_score": row["graph_score"],
            "path_type":  p["path_type"],
            "path":       " -> ".join(p["path"]),
            "path_weight": p["weight"],
        })

display(pd.DataFrame(flat_rows))

,paper_id,title,graph_score,path_type,path,path_weight
0,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,shared_topic,paper1 -> Code Generation -> paper34,0.820
1,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,shared_topic,paper1 -> Reasoning -> paper34,0.950
2,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,sibling_topic,paper1 -> Code Generation -> Large Language Mo...,0.475
3,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,sibling_topic,paper1 -> Code Generation -> Large Language Mo...,0.410
4,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,sibling_topic,paper1 -> Code Generation -> Large Language Mo...,0.410
5,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,sibling_topic,paper1 -> Reasoning -> Large Language Models -...,0.410
6,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,sibling_topic,paper1 -> Reasoning -> Large Language Models -...,0.410
7,paper34,Enhanced LLM Reasoning by Optimizing Reward Fu...,4.295,sibling_topic,paper1 -> Reasoning -> Large Language Models -...,0.410
8,paper127,SciResearcher: Scaling Deep Research Agents fo...,3.885,shared_topic,paper1 -> Reasoning -> paper127,0.950
9,paper127,SciResearcher: Scaling Deep Research Agents fo...,3.885,sibling_topic,paper1 -> Code Generation -> Large Language Mo...,0.475
